# MediTriage AI — LangGraph Agent Notebook

Run all cells to visualize the graph and test the agent.

**Workflow type:** Conditional + Iterative

In [ ]:
%pip install -q langgraph langchain langchain-groq pydantic python-dotenv

In [ ]:
from IPython.display import Image, display
from dotenv import load_dotenv

from agent import (
    SymptomAnalysis,
    RecommendationQuality,
    create_graph,
    run_assessment,
    lookup_medical_guidelines,
    calculate_urgency_score,
)

load_dotenv()

print("Imports OK")
print("SymptomAnalysis fields:", list(SymptomAnalysis.model_fields.keys()))

## Graph visualization (draw_mermaid_png)

This shows all nodes, conditional edges, and loops required by the assignment.

In [ ]:
graph = create_graph()
display(Image(graph.get_graph().draw_mermaid_png()))

## Tool demo

In [ ]:
print(lookup_medical_guidelines.invoke({"symptom": "chest pain"}))
print()
print(calculate_urgency_score.invoke({"severity": 9, "duration_days": 0.5, "red_flag_count": 2}))

## Test 1: Home care (conditional routing)

In [ ]:
result_home = run_assessment(
    graph,
    symptom="mild headache",
    duration="1 day",
    severity="3",
    other_symptoms="none",
    thread_id="notebook_home_care",
)
print("Triage level:", result_home["triage_level"])
print("Confidence:", result_home["confidence_score"])
print("Quality score:", result_home["quality_score"])
print("Iterations:", result_home["iteration_count"])
print("\n--- Recommendation preview ---\n")
print(result_home["recommendations"][:500], "...")

## Test 2: Emergency (conditional routing — different branch)

In [ ]:
result_emergency = run_assessment(
    graph,
    symptom="chest pain and difficulty breathing",
    duration="30 minutes",
    severity="9",
    other_symptoms="sweating",
    thread_id="notebook_emergency",
)
print("Triage level:", result_emergency["triage_level"])
print("Red flags:", result_emergency["red_flags"])
print("Urgency score:", result_emergency["urgency_score"])
print("\n--- Recommendation preview ---\n")
print(result_emergency["recommendations"][:500], "...")

## MemorySaver demo (thread_id persistence)

In [ ]:
config = {"configurable": {"thread_id": "notebook_emergency"}}
snapshot = graph.get_state(config)
print("Stored symptom:", snapshot.values.get("symptom"))
print("Stored triage level:", snapshot.values.get("triage_level"))
print("Assessment complete:", snapshot.values.get("assessment_complete"))